# Qwen3-VL-4B-Instruct Benchmark on AWS Neuron

This notebook benchmarks [Qwen3-VL-4B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct) on AWS Neuron hardware (Trainium2 / Inferentia2) using NxD Inference (NxDI) via the vLLM interface.

**Supported instances**: `trn2.3xlarge` (LNC=2 or LNC=1), `inf2.xlarge`

**SDK**: AWS Neuron SDK 2.28 (`Deep Learning AMI Neuron (Ubuntu 24.04) 20260227`)

**Key findings from benchmarking**:
- **tp=2 DP=2** gives highest aggregate throughput: 103.3 tok/s image+text, 116.7 tok/s text-only
- **tp=4 DP=1** gives lowest per-request latency: 91.8 tok/s image+text, 2.8s per request
- **MLP kernel hurts** at this model scale (-15.6% throughput at tp=4)
- **tp=1 is NOT viable** on SDK 2.28 (compiler bugs)
- **ISA kernels are REQUIRED** at tp=2 (without them, compiler ICE)

## Prerequisites

```bash
# Activate the pre-installed PyTorch Inference environment
source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/bin/activate

# Verify Neuron devices
neuron-ls
```

## Step 1: Configuration

Edit the cell below to configure the benchmark. The notebook auto-detects your instance type and sets safe defaults.

In [1]:
import subprocess
import os

# ============================================================
# AUTO-DETECTION (you can override below)
# ============================================================
result = subprocess.run(['neuron-ls'], capture_output=True, text=True)
output = result.stdout

# Parse instance type
instance_type = 'unknown'
if 'instance-type:' in output:
    instance_type = output.split('instance-type:')[1].split('\n')[0].strip()

# Parse core count from the table data rows (format: | 0      | 4      | 0-3      | ...)
num_cores = 0
for line in output.split('\n'):
    parts = [p.strip() for p in line.split('|') if p.strip()]
    if len(parts) >= 3 and parts[0].isdigit() and parts[1].isdigit():
        num_cores += int(parts[1])  # Sum cores across all devices

if num_cores == 0:
    # Fallback: count rows with digit-starting device IDs
    print(f"WARNING: Could not parse core count from neuron-ls. Output:\n{output}")
    num_cores = 2  # Safe default

# Parse LNC config
lnc_config = 2  # default
if 'logical-neuroncore-config:' in output:
    lnc_config = int(output.split('logical-neuroncore-config:')[1].split('\n')[0].strip())

if 'inf2' in instance_type:
    DETECTED_PLATFORM = 'inf2'
    DETECTED_CORES = num_cores
elif lnc_config == 1 or num_cores > 4:
    DETECTED_PLATFORM = 'trn2_lnc1'
    DETECTED_CORES = num_cores
else:
    DETECTED_PLATFORM = 'trn2_lnc2'
    DETECTED_CORES = num_cores

print(f"Detected: {instance_type}, {num_cores} cores, LNC={lnc_config} -> platform={DETECTED_PLATFORM}")

# ============================================================
# USER CONFIGURATION - Edit these values
# ============================================================

# Model
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"  # HuggingFace model ID
MODEL_PATH = os.path.expanduser("~/models/Qwen3-VL-4B-Instruct")  # Local download path (also check /mnt/models/)
if os.path.exists("/mnt/models/Qwen3-VL-4B-Instruct"):
    MODEL_PATH = "/mnt/models/Qwen3-VL-4B-Instruct"  # Use existing model on NVMe

# Parallelism
TP_DEGREE = 4       # Tensor parallel degree: 2 or 4 (tp=1 NOT supported on SDK 2.28)
                     # tp=2: 61.7 tok/s single-req, enables DP=2
                     # tp=4: 95.1 tok/s single-req, uses all cores on LNC=2

# Sequence length
SEQ_LEN = 4096      # Max sequence length: 1024, 2048, 4096, or 8192

# ISA Kernels
QKV_KERNEL_ENABLED = True   # QKV fused kernel (REQUIRED at tp=2, recommended at tp=4)
ATTN_KERNEL_ENABLED = True  # Attention kernel (REQUIRED at tp=2, recommended at tp=4)
MLP_KERNEL_ENABLED = False  # MLP kernel (only works at tp>=4, HURTS performance at this scale)

# Generation
MAX_NEW_TOKENS = 256  # Max tokens to generate per request

# Benchmark
WARMUP_RUNS = 2       # Number of warmup iterations (discarded)
BENCHMARK_RUNS = 5    # Number of timed iterations

# LNC setting (trn2 only)
LNC_MODE = 2  # 1 or 2 (must match /etc/environment NEURON_LOGICAL_NC_CONFIG)

Detected: trn2.3xlarge, 4 cores, LNC=2 -> platform=trn2_lnc2


In [2]:
# ============================================================
# VALIDATION - Checks for incompatible configurations
# ============================================================

errors = []
warnings = []

# tp=1 is not supported
if TP_DEGREE == 1:
    errors.append(
        "tp=1 is NOT supported on SDK 2.28 for Qwen3-VL-4B. "
        "QKV kernel I=6144 > 4096 ISA limit (with kernels), "
        "compiler ICE NCC_IBIR182 (without kernels). Use tp=2 or tp=4."
    )

# tp must not exceed available cores
if TP_DEGREE > DETECTED_CORES:
    errors.append(
        f"tp={TP_DEGREE} exceeds available cores ({DETECTED_CORES}). "
        f"Max TP for {DETECTED_PLATFORM} is {DETECTED_CORES}."
    )

# inf2: ISA kernels must be off
if DETECTED_PLATFORM == 'inf2':
    if QKV_KERNEL_ENABLED or ATTN_KERNEL_ENABLED or MLP_KERNEL_ENABLED:
        warnings.append(
            "ISA kernels are NOT supported on inf2 (compiler bug NCC_IBVF032). Disabling all kernels."
        )
        QKV_KERNEL_ENABLED = False
        ATTN_KERNEL_ENABLED = False
        MLP_KERNEL_ENABLED = False
    # inf2 + tp=2 + no kernels may trigger NCC_ITEN404 (needs testing)
    warnings.append(
        "inf2 with kernels off may trigger compiler ICE NCC_ITEN404. "
        "This has been observed on trn2 but may not reproduce on inf2."
    )

# trn2 tp=2: kernels are REQUIRED
if DETECTED_PLATFORM.startswith('trn2') and TP_DEGREE == 2:
    if not QKV_KERNEL_ENABLED or not ATTN_KERNEL_ENABLED:
        warnings.append(
            "QKV and Attention kernels are REQUIRED at tp=2 on trn2 (compiler ICE NCC_ITEN404 without them). Enabling."
        )
        QKV_KERNEL_ENABLED = True
        ATTN_KERNEL_ENABLED = True

# MLP kernel size check
INTERMEDIATE_SIZE = 9728
if MLP_KERNEL_ENABLED and INTERMEDIATE_SIZE // TP_DEGREE > 4096:
    warnings.append(
        f"MLP kernel requires intermediate_size/tp <= 4096. "
        f"{INTERMEDIATE_SIZE}/{TP_DEGREE} = {INTERMEDIATE_SIZE // TP_DEGREE} > 4096. Disabling MLP kernel."
    )
    MLP_KERNEL_ENABLED = False

# MLP kernel performance warning at tp=4
if MLP_KERNEL_ENABLED and TP_DEGREE == 4:
    warnings.append(
        "MLP kernel HURTS performance at tp=4 for this model (-15.6% throughput). "
        "Consider setting MLP_KERNEL_ENABLED = False."
    )

# LNC mode check
if DETECTED_PLATFORM.startswith('trn2'):
    expected_cores_lnc2 = 4  # trn2.3xlarge
    expected_cores_lnc1 = 8
    if LNC_MODE == 2 and DETECTED_CORES != expected_cores_lnc2:
        warnings.append(f"LNC_MODE=2 but detected {DETECTED_CORES} cores (expected {expected_cores_lnc2}). Check NEURON_LOGICAL_NC_CONFIG.")
    if LNC_MODE == 1 and DETECTED_CORES != expected_cores_lnc1:
        warnings.append(f"LNC_MODE=1 but detected {DETECTED_CORES} cores (expected {expected_cores_lnc1}). Check NEURON_LOGICAL_NC_CONFIG.")

# Print results
for e in errors:
    print(f"ERROR: {e}")
for w in warnings:
    print(f"WARNING: {w}")

if errors:
    raise ValueError("Fix the above errors before proceeding.")

print(f"\nConfiguration validated:")
print(f"  Platform:    {DETECTED_PLATFORM} ({DETECTED_CORES} cores)")
print(f"  TP degree:   {TP_DEGREE}")
print(f"  Seq length:  {SEQ_LEN}")
print(f"  QKV kernel:  {QKV_KERNEL_ENABLED}")
print(f"  Attn kernel: {ATTN_KERNEL_ENABLED}")
print(f"  MLP kernel:  {MLP_KERNEL_ENABLED}")
print(f"  Max tokens:  {MAX_NEW_TOKENS}")


Configuration validated:
  Platform:    trn2_lnc2 (4 cores)
  TP degree:   4
  Seq length:  4096
  QKV kernel:  True
  Attn kernel: True
  MLP kernel:  False
  Max tokens:  256


## Step 2: Install Dependencies and Download Model

In [3]:
!pip install -q qwen-vl-utils pillow

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [4]:
import time

if not os.path.exists(MODEL_PATH):
    print(f"Downloading {MODEL_ID} to {MODEL_PATH}...")
    from huggingface_hub import snapshot_download
    t0 = time.time()
    snapshot_download(MODEL_ID, local_dir=MODEL_PATH)
    print(f"Download complete in {time.time() - t0:.1f}s")
else:
    print(f"Model already exists at {MODEL_PATH}")

Model already exists at /mnt/models/Qwen3-VL-4B-Instruct


## Step 3: Build Neuron Configuration

The configuration is auto-generated from your settings above. Two separate configs are needed:
- **text_neuron_config**: For the autoregressive text model (context encoding + token generation)
- **vision_neuron_config**: For the vision encoder (no ISA kernels, different bucket structure)

In [5]:
# Build bucket lists based on SEQ_LEN
def make_buckets(seq_len):
    """Generate appropriate bucket list for the given seq_len."""
    candidates = [512, 1024, 2048, 4096, 8192]
    return [b for b in candidates if b <= seq_len]

text_buckets = make_buckets(SEQ_LEN)
vision_buckets = [b for b in [1024, 4096] if b <= SEQ_LEN]

text_neuron_config = {
    "batch_size": 1,
    "ctx_batch_size": 1,
    "tkg_batch_size": 1,
    "seq_len": SEQ_LEN,
    "max_context_length": SEQ_LEN,
    "torch_dtype": "bfloat16",
    "tp_degree": TP_DEGREE,
    "world_size": TP_DEGREE,
    "enable_bucketing": True,
    "context_encoding_buckets": text_buckets,
    "token_generation_buckets": text_buckets,
    "fused_qkv": True,
    "qkv_kernel_enabled": QKV_KERNEL_ENABLED,
    "mlp_kernel_enabled": MLP_KERNEL_ENABLED,
    "attn_kernel_enabled": ATTN_KERNEL_ENABLED,
    "logical_neuron_cores": LNC_MODE,
    "cc_pipeline_tiling_factor": LNC_MODE,
    "rpl_reduce_dtype": "bfloat16",
    "attention_dtype": "bfloat16",
    "cast_type": "as-declared",
}

vision_neuron_config = {
    "batch_size": 1,
    "seq_len": min(SEQ_LEN, 4096),  # Vision encoder caps at 4096
    "max_context_length": min(SEQ_LEN, 4096),
    "enable_bucketing": True,
    "buckets": vision_buckets,
    "world_size": TP_DEGREE,
    "tp_degree": TP_DEGREE,
    "torch_dtype": "bfloat16",
    "rpl_reduce_dtype": "bfloat16",
    "cast_type": "as-declared",
    "logical_neuron_cores": LNC_MODE,
    "cc_pipeline_tiling_factor": LNC_MODE,
    "fused_qkv": True,
    "attn_kernel_enabled": False,  # Vision encoder: ISA kernels not supported
    "mlp_kernel_enabled": False,
}

print("Text neuron config:")
for k, v in text_neuron_config.items():
    print(f"  {k}: {v}")
print(f"\nVision neuron config:")
for k, v in vision_neuron_config.items():
    print(f"  {k}: {v}")

Text neuron config:
  batch_size: 1
  ctx_batch_size: 1
  tkg_batch_size: 1
  seq_len: 4096
  max_context_length: 4096
  torch_dtype: bfloat16
  tp_degree: 4
  world_size: 4
  enable_bucketing: True
  context_encoding_buckets: [512, 1024, 2048, 4096]
  token_generation_buckets: [512, 1024, 2048, 4096]
  fused_qkv: True
  qkv_kernel_enabled: True
  mlp_kernel_enabled: False
  attn_kernel_enabled: True
  logical_neuron_cores: 2
  cc_pipeline_tiling_factor: 2
  rpl_reduce_dtype: bfloat16
  attention_dtype: bfloat16
  cast_type: as-declared

Vision neuron config:
  batch_size: 1
  seq_len: 4096
  max_context_length: 4096
  enable_bucketing: True
  buckets: [1024, 4096]
  world_size: 4
  tp_degree: 4
  torch_dtype: bfloat16
  rpl_reduce_dtype: bfloat16
  cast_type: as-declared
  logical_neuron_cores: 2
  cc_pipeline_tiling_factor: 2
  fused_qkv: True
  attn_kernel_enabled: False
  mlp_kernel_enabled: False


## Step 4: Apply Patches and Compile Model

The 4B model has `tie_word_embeddings=true`, which crashes SDK 2.28 during weight loading.
We apply a monkey-patch to copy `embed_tokens.weight` → `lm_head.weight` before compilation.

**Note**: First compilation takes 5-10 minutes. Subsequent runs use cached NEFFs (~30-90s).

In [6]:
# Apply tie_word_embeddings patch for 4B model
from neuronx_distributed_inference.models.qwen3_vl.modeling_qwen3_vl import (
    NeuronQwen3VLForCausalLM,
)

@staticmethod
def update_state_dict_for_tied_weights(state_dict):
    if "lm_head.weight" not in state_dict and "embed_tokens.weight" in state_dict:
        print("[PATCH] tie_word_embeddings: copying embed_tokens -> lm_head")
        state_dict["lm_head.weight"] = state_dict["embed_tokens.weight"].clone()

NeuronQwen3VLForCausalLM.update_state_dict_for_tied_weights = update_state_dict_for_tied_weights
print("Patched: tie_word_embeddings handler for 4B model")

/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/parallel_layers/layers.py:16: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/parallel_layers/layers.py:16: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/parallel_layers/layers.py:16: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (


/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:74: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:74: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:74: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:74: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx

/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/moe_fused_tkg.py:49: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/moe_fused_tkg.py:49: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)


/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed_inference/modules/attention/utils.py:13: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from neuronx_distributed_inference.modules.custom_calls import neuron_cumsum
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed_inference/modules/lora_serving/lora_checkpoint.py:9: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from neuronx_distributed_inference.modules.attention.gqa import replicate_kv
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed_inference/modules/lora_serving/lora_checkpoint.py:9: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from neuronx_distributed_inference.modules.attention.gqa import replicate_kv
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuron

/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed_inference/models/llama/modeling_llama.py:66: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from neuronx_distributed_inference.modules.attention.attention_base import NeuronAttentionBase
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed_inference/models/llama/modeling_llama.py:66: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from neuronx_distributed_inference.modules.attention.attention_base import NeuronAttentionBase
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed_inference/models/llama/modeling_llama.py:66: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from neuronx_distributed_inference.modules.attention.attention_base import NeuronAttentionBase
/opt/aws_neuronx_venv_pytorch_inference_vll

Patched: tie_word_embeddings handler for 4B model


In [7]:
os.environ["VLLM_NEURON_FRAMEWORK"] = "neuronx-distributed-inference"

from vllm import LLM, SamplingParams

print(f"Compiling model with tp={TP_DEGREE}, seq_len={SEQ_LEN}...")
print(f"Kernels: QKV={QKV_KERNEL_ENABLED}, Attn={ATTN_KERNEL_ENABLED}, MLP={MLP_KERNEL_ENABLED}")

compile_start = time.time()
llm = LLM(
    model=MODEL_PATH,
    tokenizer=MODEL_PATH,
    trust_remote_code=True,
    dtype="bfloat16",
    tensor_parallel_size=TP_DEGREE,
    max_num_seqs=1,
    max_model_len=SEQ_LEN,
    additional_config=dict(
        override_neuron_config=dict(
            text_neuron_config=text_neuron_config,
            vision_neuron_config=vision_neuron_config,
        )
    ),
    limit_mm_per_prompt={"image": 1},
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
)
compile_time = time.time() - compile_start
print(f"\nModel ready in {compile_time:.1f}s")

INFO 03-12 18:18:49 [__init__.py:43] Available plugins for group vllm.platform_plugins:


INFO 03-12 18:18:49 [__init__.py:45] - neuron -> vllm_neuron:register


INFO 03-12 18:18:49 [__init__.py:48] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.


INFO 03-12 18:18:49 [__init__.py:217] Platform plugin neuron is activated


INFO 03-12 18:18:49 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.


...

INFO:vllm_neuron.platform:The custom Neuron scheduler will disable chunked prefill and schedule requests using the continuous batching mechanism, prioritizing prefill over decode.


(EngineCore_DP0 pid=36401) 

INFO:vllm_neuron.platform:Neuron custom scheduler default: max_num_batched_tokens set to 131072. Override with --max-num-batched-tokens if needed.


INFO 03-12 18:24:35 [llm.py:360] Supported tasks: ['generate']



Model ready in 345.0s


## Step 5: Prepare Sample Images

Images are resized to stay within the vision encoder's patch budget (max 4096 visual tokens).
Each 56×56 pixel block becomes one visual token after the spatial merge.

In [8]:
import urllib.request
from io import BytesIO
from PIL import Image

MAX_LONG_SIDE = 1024  # Keep images within 4096-token vision budget

def load_and_resize(source, max_long_side=MAX_LONG_SIDE):
    """Load image from URL or path and resize to fit within patch budget."""
    if source.startswith('http'):
        req = urllib.request.Request(source, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            img = Image.open(BytesIO(resp.read())).convert("RGB")
    else:
        img = Image.open(source).convert("RGB")

    w, h = img.size
    if max(w, h) > max_long_side:
        scale = max_long_side / max(w, h)
        new_w = int(w * scale) // 56 * 56  # Align to patch grid (56 = patch_size * spatial_merge)
        new_h = int(h * scale) // 56 * 56
        new_w = max(new_w, 56)
        new_h = max(new_h, 56)
        img = img.resize((new_w, new_h), Image.LANCZOS)
        print(f"  Resized: {w}x{h} -> {new_w}x{new_h} (~{new_w * new_h // 56 // 56} visual tokens)")
    else:
        print(f"  Size: {w}x{h} (~{w * h // 56 // 56} visual tokens)")
    return img

# Load sample images
print("Loading sample images...")
print("Image 1 (Qwen demo - woman with dog on beach):")
img1 = load_and_resize("https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg")

print("Image 2 (landscape):")
try:
    img2 = load_and_resize("https://picsum.photos/id/24/800/600")
except Exception as e:
    print(f"  Failed to load ({e}), using synthetic image")
    img2 = Image.new("RGB", (672, 504), color=(100, 150, 200))

print("Done.")

Loading sample images...
Image 1 (Qwen demo - woman with dog on beach):


  Resized: 2048x1365 -> 1008x672 (~216 visual tokens)
Image 2 (landscape):


  Size: 800x600 (~153 visual tokens)
Done.


## Step 6: Inference Helper

In [9]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)

def run_inference(llm, prompt_text, image=None, max_tokens=MAX_NEW_TOKENS):
    """Run inference and return result dict with timing."""
    if image is not None:
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text},
            ],
        }]
    else:
        messages = [{"role": "user", "content": prompt_text}]

    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    if image is not None:
        inputs = {"prompt": prompt, "multi_modal_data": {"image": [image]}}
    else:
        inputs = {"prompt": prompt}

    sampling_params = SamplingParams(top_k=1, max_tokens=max_tokens, temperature=0.0)

    t0 = time.time()
    outputs = llm.generate([inputs], sampling_params=sampling_params)
    latency = time.time() - t0

    text = outputs[0].outputs[0].text
    num_tokens = len(outputs[0].outputs[0].token_ids)
    tok_per_sec = num_tokens / latency if latency > 0 else 0

    return {
        "text": text,
        "num_tokens": num_tokens,
        "latency_s": latency,
        "tok_per_sec": tok_per_sec,
    }

print("Inference helper ready.")

Inference helper ready.


## Step 7: Example — Text-Only Inference

In [10]:
print("=" * 60)
print("TEXT-ONLY INFERENCE")
print("=" * 60)

text_prompt = "Explain what a vision-language model is and how it works. Be concise."
print(f"Prompt: {text_prompt}\n")

result = run_inference(llm, text_prompt)
print(f"Response ({result['num_tokens']} tokens, {result['latency_s']:.2f}s, {result['tok_per_sec']:.1f} tok/s):")
print(result['text'][:2000])

TEXT-ONLY INFERENCE
Prompt: Explain what a vision-language model is and how it works. Be concise.



Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Response (95 tokens, 0.95s, 99.6 tok/s):
A vision-language model (VLM) is a type of artificial intelligence model that understands and generates text based on visual input (like images) and vice versa. It combines vision and language capabilities, typically using transformers, to map images to text descriptions (e.g., “a cat sitting on a couch”) or generate images from text prompts. VLMs learn from large datasets of image-text pairs, enabling them to reason about visual content and communicate it in natural language.


## Step 8: Example — Image + Text Inference

In [11]:
print("=" * 60)
print("IMAGE + TEXT INFERENCE")
print("=" * 60)

# Image description
img_prompt = "Describe this image in detail. What do you see?"
print(f"\nImage 1: Qwen demo (woman with dog)")
print(f"Prompt: {img_prompt}\n")

result = run_inference(llm, img_prompt, image=img1)
print(f"Response ({result['num_tokens']} tokens, {result['latency_s']:.2f}s, {result['tok_per_sec']:.1f} tok/s):")
print(result['text'][:2000])

IMAGE + TEXT INFERENCE

Image 1: Qwen demo (woman with dog)
Prompt: Describe this image in detail. What do you see?



Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Response (256 tokens, 4.70s, 54.5 tok/s):
This is a heartwarming, sun-drenched photograph capturing a tender moment between a woman and her dog on a beach at sunset.

**Central Subjects:**
- A large, light-colored Labrador Retriever is sitting on the sand, facing the woman. The dog is wearing a colorful, patterned harness with a red and blue design. It is extending its front left paw toward the woman in a playful, high-five gesture.
- A young woman with long, dark hair is sitting on the sand, facing the dog. She is wearing a plaid shirt (dark blue and white or black and white) with the sleeves rolled up, and dark pants. She is barefoot. She is smiling warmly and joyfully, looking at the dog as she reciprocates the high-five with her own hand.

**Setting and Atmosphere:**
- The scene is set on a wide, sandy beach. The sand is textured with footprints and gentle ripples, and the woman and dog are sitting close together.
- In the background, the ocean stretches to the horizon, with soft, 

In [12]:
# Visual Q&A
qa_prompt = "How many people are in this image? What are they doing?"
print(f"\nImage 2: Landscape")
print(f"Prompt: {qa_prompt}\n")

result = run_inference(llm, qa_prompt, image=img2)
print(f"Response ({result['num_tokens']} tokens, {result['latency_s']:.2f}s, {result['tok_per_sec']:.1f} tok/s):")
print(result['text'][:2000])


Image 2: Landscape
Prompt: How many people are in this image? What are they doing?



Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Response (78 tokens, 1.01s, 77.5 tok/s):
There are no people in this image.

The image shows an open book resting on a wooden surface. The book’s pages are filled with text, and the focus is on the spine and the edges of the pages. The background is softly blurred, suggesting a calm, quiet setting, perhaps for reading or study. Since there are no human figures visible, there is no activity involving people.


## Step 9: Benchmark Suite

Runs timed iterations for both text-only and image+text prompts.
Reports mean, std, P50, P99 latency and throughput.

In [13]:
import numpy as np

def run_benchmark(llm, prompt_text, image=None, warmup=WARMUP_RUNS, runs=BENCHMARK_RUNS, label=""):
    """Run warmup + timed benchmark iterations."""
    print(f"\n{'=' * 60}")
    print(f"BENCHMARK: {label}")
    print(f"  Warmup: {warmup} runs, Benchmark: {runs} runs")
    print(f"{'=' * 60}")

    # Warmup
    for i in range(warmup):
        r = run_inference(llm, prompt_text, image=image)
        print(f"  Warmup {i+1}: {r['latency_s']:.2f}s, {r['tok_per_sec']:.1f} tok/s, {r['num_tokens']} tokens")

    # Timed runs
    latencies = []
    throughputs = []
    token_counts = []

    for i in range(runs):
        r = run_inference(llm, prompt_text, image=image)
        latencies.append(r['latency_s'])
        throughputs.append(r['tok_per_sec'])
        token_counts.append(r['num_tokens'])
        print(f"  Run {i+1}: {r['latency_s']:.2f}s, {r['tok_per_sec']:.1f} tok/s, {r['num_tokens']} tokens")

    lat = np.array(latencies)
    tp = np.array(throughputs)
    toks = np.array(token_counts)

    results = {
        "label": label,
        "latency_mean": float(lat.mean()),
        "latency_std": float(lat.std()),
        "latency_p50": float(np.percentile(lat, 50)),
        "latency_p99": float(np.percentile(lat, 99)),
        "throughput_mean": float(tp.mean()),
        "throughput_std": float(tp.std()),
        "throughput_min": float(tp.min()),
        "throughput_max": float(tp.max()),
        "tokens_mean": float(toks.mean()),
    }

    print(f"\n  --- Results ---")
    print(f"  Throughput: {results['throughput_mean']:.1f} tok/s (std={results['throughput_std']:.1f}, min={results['throughput_min']:.1f}, max={results['throughput_max']:.1f})")
    print(f"  Latency:   {results['latency_mean']:.3f}s (std={results['latency_std']:.3f}, P50={results['latency_p50']:.3f}, P99={results['latency_p99']:.3f})")
    print(f"  Tokens:    {results['tokens_mean']:.0f} mean")

    return results

print("Benchmark function ready.")

Benchmark function ready.


In [14]:
# Run text-only benchmark
text_benchmark_prompt = "Explain what a vision-language model is and how it works. Be concise."
text_results = run_benchmark(llm, text_benchmark_prompt, label="Text-Only")


BENCHMARK: Text-Only
  Warmup: 2 runs, Benchmark: 5 runs


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Warmup 1: 0.92s, 102.9 tok/s, 95 tokens


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Run 4: 0.93s, 102.2 tok/s, 95 tokens


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Run 5: 0.93s, 101.6 tok/s, 95 tokens

  --- Results ---
  Throughput: 102.1 tok/s (std=0.6, min=101.3, max=103.0)
  Latency:   0.930s (std=0.006, P50=0.930, P99=0.938)
  Tokens:    95 mean


In [15]:
# Run image+text benchmark (using image 1 for consistency)
img_benchmark_prompt = "Describe this image in detail. What do you see?"
img_results = run_benchmark(llm, img_benchmark_prompt, image=img1, label="Image+Text")


BENCHMARK: Image+Text
  Warmup: 2 runs, Benchmark: 5 runs


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Warmup 1: 2.73s, 93.8 tok/s, 256 tokens


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Run 4: 2.72s, 94.1 tok/s, 256 tokens


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Run 5: 2.70s, 94.9 tok/s, 256 tokens

  --- Results ---
  Throughput: 94.2 tok/s (std=0.5, min=93.4, max=94.9)
  Latency:   2.719s (std=0.014, P50=2.721, P99=2.739)
  Tokens:    256 mean


## Step 10: Results Summary

In [16]:
print("\n" + "=" * 70)
print("BENCHMARK RESULTS SUMMARY")
print("=" * 70)
print(f"\nConfiguration:")
print(f"  Instance:    {instance_type} ({DETECTED_PLATFORM})")
print(f"  TP degree:   {TP_DEGREE}")
print(f"  Seq length:  {SEQ_LEN}")
print(f"  LNC mode:    {LNC_MODE}")
print(f"  QKV kernel:  {QKV_KERNEL_ENABLED}")
print(f"  Attn kernel: {ATTN_KERNEL_ENABLED}")
print(f"  MLP kernel:  {MLP_KERNEL_ENABLED}")
print(f"  Max tokens:  {MAX_NEW_TOKENS}")
print(f"  Compile time: {compile_time:.1f}s")

print(f"\n{'Metric':<30} {'Text-Only':>12} {'Image+Text':>12}")
print("-" * 56)
print(f"{'Throughput (tok/s)':<30} {text_results['throughput_mean']:>11.1f}  {img_results['throughput_mean']:>11.1f}")
print(f"{'  std':<30} {text_results['throughput_std']:>11.1f}  {img_results['throughput_std']:>11.1f}")
print(f"{'  min':<30} {text_results['throughput_min']:>11.1f}  {img_results['throughput_min']:>11.1f}")
print(f"{'  max':<30} {text_results['throughput_max']:>11.1f}  {img_results['throughput_max']:>11.1f}")
print(f"{'Latency (s)':<30} {text_results['latency_mean']:>11.3f}  {img_results['latency_mean']:>11.3f}")
print(f"{'  P50':<30} {text_results['latency_p50']:>11.3f}  {img_results['latency_p50']:>11.3f}")
print(f"{'  P99':<30} {text_results['latency_p99']:>11.3f}  {img_results['latency_p99']:>11.3f}")
print(f"{'Avg tokens generated':<30} {text_results['tokens_mean']:>11.0f}  {img_results['tokens_mean']:>11.0f}")


BENCHMARK RESULTS SUMMARY

Configuration:
  Instance:    trn2.3xlarge (trn2_lnc2)
  TP degree:   4
  Seq length:  4096
  LNC mode:    2
  QKV kernel:  True
  Attn kernel: True
  MLP kernel:  False
  Max tokens:  256
  Compile time: 345.0s

Metric                            Text-Only   Image+Text
--------------------------------------------------------
Throughput (tok/s)                   102.1         94.2
  std                                  0.6          0.5
  min                                101.3         93.4
  max                                103.0         94.9
Latency (s)                          0.930        2.719
  P50                                0.930        2.721
  P99                                0.938        2.739
Avg tokens generated                    95          256


In [17]:
# Save results to JSON
import json

all_results = {
    "config": {
        "instance_type": instance_type,
        "platform": DETECTED_PLATFORM,
        "num_cores": DETECTED_CORES,
        "tp_degree": TP_DEGREE,
        "seq_len": SEQ_LEN,
        "lnc_mode": LNC_MODE,
        "qkv_kernel": QKV_KERNEL_ENABLED,
        "attn_kernel": ATTN_KERNEL_ENABLED,
        "mlp_kernel": MLP_KERNEL_ENABLED,
        "max_new_tokens": MAX_NEW_TOKENS,
        "compile_time_s": compile_time,
    },
    "text_only": text_results,
    "image_text": img_results,
}

results_file = f"results_{DETECTED_PLATFORM}_tp{TP_DEGREE}_seq{SEQ_LEN}.json"
with open(results_file, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"Results saved to {results_file}")

Results saved to results_trn2_lnc2_tp4_seq4096.json


## Reference: Benchmark Results (trn2.3xlarge, LNC=2, SDK 2.28)

These results were collected on 2026-03-12 with SDK 2.28, Qwen3-VL-4B-Instruct, `max_new_tokens=256`, `seq_len=4096`.

### NxDI Offline (via vLLM LLM class, single request)

| TP | Kernels | Image+Text tok/s | Text-Only tok/s | Compile (s) | Notes |
|----|---------|-----------------|----------------|-------------|-------|
| 2 | QKV+Attn | 61.7 | 65.7 | 367 | Baseline |
| 4 | QKV+Attn+MLP | 82.3 | 88.8 | 279 | MLP kernel works but hurts |
| **4** | **QKV+Attn** | **95.1** | **103.5** | 325 | **Fastest single-request** |
| 1 | QKV+Attn | FAIL | FAIL | -- | QKV I=6144 > 4096 ISA limit |
| 2 | None | FAIL | FAIL | -- | Compiler ICE NCC_ITEN404 |
| 1 | None | FAIL | FAIL | -- | Compiler ICE NCC_IBIR182 |

### vLLM Serve (via OpenAI API, aggregate throughput)

| TP | DP | Concurrent | Image+Text tok/s | Text-Only tok/s | Per-Req Latency (img) |
|----|-----|-----------|-----------------|----------------|---------------------|
| 4 | 1 | 1 | 91.8 | 103.1 | 2.79s |
| 2 | 1 | 1 | 59.8 | 65.5 | 4.28s |
| **2** | **2** | **2** | **103.3** | **116.7** | 4.94s |
| 4 | 1 | 2 | FAIL | -- | -- |
| 2 | 2 | 4 | FAIL | -- | -- |

### Key Findings

1. **For maximum throughput (serving)**: Use **tp=2 DP=2** via vLLM serve. 103.3 img tok/s, 116.7 text tok/s. This is +12.5% / +13.2% over tp=4 DP=1 using the same 4 cores.

2. **For minimum latency (interactive)**: Use **tp=4 DP=1**. 2.8s per image request (vs 4.9s with tp=2). 95.1 tok/s single-request via NxDI offline.

3. **MLP kernel hurts performance**: At tp=4, enabling MLP kernel costs -15.6% throughput. The ISA dispatch overhead exceeds compute benefit when `intermediate_size/4 = 2432`.

4. **ISA kernels are mandatory at tp=2**: Without QKV+Attn kernels, the compiler hits ICE NCC_ITEN404. They route around buggy compiler code paths.

5. **tp=1 is blocked**: Two independent compiler bugs prevent tp=1 on SDK 2.28.

6. **bs=1 limits concurrency**: With batch_size=1 compiled model, each DP worker handles exactly 1 request. Sending more concurrent requests than DP workers causes 500 errors. True batching requires recompiling with higher batch_size.

7. **vLLM serving overhead is minimal**: ~3% for image+text, <1% for text-only vs NxDI offline.

## Reference: Configuration Guide

### Which config should I use?

| Scenario | TP | DP | Framework | Expected Performance |
|----------|----|----|-----------|---------------------|
| **Serving API (throughput)** | 2 | 2 | vLLM serve | ~103 img tok/s, ~117 text tok/s |
| **Interactive (latency)** | 4 | 1 | NxDI offline or vLLM | ~95 img tok/s, ~103 text tok/s, 2.8s/req |
| **Simple testing** | 2 | 1 | NxDI offline | ~62 img tok/s, ~66 text tok/s |

### Instance selection

| Instance | Best For | Cores | Notes |
|----------|----------|-------|-------|
| trn2.3xlarge LNC=2 | Best performance | 4 | tp=2 DP=2 or tp=4 DP=1 |
| trn2.3xlarge LNC=1 | More DP workers | 8 | Not yet benchmarked |
| inf2.xlarge | Cost-effective | 2 | ISA kernels off, may hit compiler bugs |

### Known SDK 2.28 limitations for this model

- `tie_word_embeddings=true` crashes weight loading (patched in this notebook)
- tp=1 not viable (QKV kernel limit + compiler ICE)
- Kernels-off at tp=2 triggers compiler ICE NCC_ITEN404
- MLP kernel overhead exceeds benefit at this model scale
- batch_size > 1 requires Chandra 3-bug NxDI patch
- vLLM DP > 1 requires pre-compilation workaround (compile race condition)
- Image must be resized client-side for vLLM serve (no automatic resize for patch budget)

## Cleanup

If you're done benchmarking, remember to:
1. Save any results you need
2. Stop or terminate your instance to avoid unnecessary charges

```bash
# Check for running Neuron processes
neuron-top
```